# LM2500 — Tier C: Two-mass mechanical shaft + voltage-sensitive load

## Overview — what this notebook adds beyond Tier A/B

Tier A and Tier B both treat the LM2500 as a single lumped rotational mass (combined free-power-turbine + generator inertia, `H ≈ 2.8 s`). That's adequate for grid-side frequency studies, but it hides a real dynamic: the LM2500 is a **two-shaft engine**. The gas generator (HP compressor + HP turbine on a single spool) runs at variable speed — about 4,900 rpm at idle and 9,500 rpm at full power — and is mechanically *decoupled* from the power turbine. The power turbine itself runs at the constant 3,600 rpm needed by the generator.

These two rotors are linked only through the **gas path**: combustion gas leaves the HP turbine, expands through the IPT/FPT, and delivers shaft power to the PT. There is no mechanical shaft between HP and PT. During a load transient, the HP rotor spools up or down with its own (small) inertia and its own (~0.5-1 s) time constant, *before* the additional gas energy reaches the PT and accelerates the generator.

**What Tier C resolves that Tier A/B cannot:**

1. **HP-rotor speed transient.** The single-mass models implicitly assume the gas-path response is instantaneous. Tier C makes it explicit.
2. **Two distinct timescales.** `H_hp ≈ 0.3 s` is much smaller than `H_pt ≈ 2.5 s`. The HP rotor reaches a new operating point ~5× faster than the PT/gen rotor.
3. **Voltage-sensitive load** via a ZIP exponent `Pe = demand · (V/Vref)^pv · (1 + α(ω−1))`. V is held at 1 pu here (pure scipy path, perfect AVR assumption). For real V-Q dynamics with GENROU + EXST1, use the ANDES path in `gas_plant_andes/`.

**Validation strategy:** SS hold (sanity), 15→18 MW step (HP rotor visibility), pv-exponent sweep, full 7.4-hr Load17 comparison vs Tier 0/A/B.

### Cell 1 — Imports

Pulls in `simulate_multishaft` (this tier's main entry point) plus `simulate_tgov1` and `simulate_ggov1` for direct comparison against Tier A and Tier B. The `GasTurbinePlant` surrogate (scaled to 22 MW) supplies the steady-state fuel/efficiency table used by `dispatch_fn`.

*Expected:* `Imports OK` and nothing else.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from gas_plant import GasTurbinePlant
from gas_plant.dynamics import (
    TGOV1Params, simulate_tgov1,
    GGOV1Params, simulate_ggov1,
    MultishaftParams, simulate_multishaft,
)

lm2500 = GasTurbinePlant(rated_power_mw=22.0)
dispatch_fn = lambda lf: lm2500.dispatch(lf)
print('Imports OK')

## 1. Parameter set

### Cell 2 — explanation

`MultishaftParams` is composition-based: it carries a full `GGOV1Params` instance (Tier B governor) plus the new mechanical/load fields.

**Mechanical split (per LM2500 physical configuration):**

| Symbol | Default | Meaning |
|---|---|---|
| `H_pt_s` | 2.5 s | PT + generator rotor inertia (the slow heavy mass) |
| `H_hp_s` | 0.3 s | HP rotor inertia (small, fast) |
| `D_pt` | 1.0 pu/pu | PT damping referenced to 60 Hz (correct — synchronous reference) |
| `D_hp` | **0.0** | HP rotor damping — see below |

**Why `D_hp = 0`:** the standard swing-equation damping form `D·(ω − 1)` assumes the rotor's natural rest speed is synchronous (60 Hz, ω=1 pu). For the HP rotor that is **wrong** — its natural speed is whatever the gas-path balance dictates (~7,900 rpm at 15 MW). A nonzero `D_hp` would unphysically pull the HP rotor toward 60 Hz at steady state, forcing the governor to compensate by dropping the valve. Earlier in development this exact bug shaved ~3.9 MW off the SS power. The fix is to let the gas-path coupling `P_couple(ω_hp)` provide the effective damping naturally.

**Gas-path coupling:** `P_couple = K_couple · (ω_hp − ω_hp_idle)`, where `ω_hp_idle = 4,900/9,500 = 0.516`. `K_couple = 1/(1 − 0.516) = 2.066` is set so that ω_hp = 1.0 (full NGG) delivers full pu power to the PT.

**Voltage exponent:** Tier C accepts a ZIP `pv_exponent`. With `V_term_pu = 1.0` and `pv = 0` (defaults), Tier C reproduces Tier B's constant-P load — useful as a baseline. Setting `pv = 2` would model a constant-impedance load (motor-heavy), which is more frequency-friendly (Pe drops with V drop).

*Expected output:* the parameter values listed in the dataclass.

In [ ]:
p = MultishaftParams()
print(f'H_pt = {p.H_pt_s} s, H_hp = {p.H_hp_s} s, total H = {p.H_pt_s + p.H_hp_s} s')
print(f'D_pt = {p.D_pt}, D_hp = {p.D_hp}')
print(f'omega_hp_idle = {p.omega_hp_idle}, K_couple = {p.K_couple:.3f}')
print(f'pv_exponent = {p.pv_exponent}, V_term = {p.V_term_pu}')

## 2. Steady-state hold — fundamental sanity check

### Cell 3 — explanation

Hold demand at a constant 15 MW for 30 s with the model at its self-consistent IC. If the model can't hold a static operating point with zero drift, every dynamic result that follows is suspect.

**What's being solved:** the 14-state ODE under constant `Pe_demand`, starting from `_initial_state_for_load(Pe0_pu)` which solves the algebraic steady-state equations.

**What to expect:**
- `df = 0.0000 mHz` (to machine precision)
- `omega_pt = 1.000000` (no PT frequency drift)
- `omega_hp ≈ 0.832` pu of NGG full → `~7,901 rpm` (the physical gas-generator speed at 15 MW load, halfway between NGG_idle 4,900 and the value at full 22 MW)
- `Pm_hp = Pm_pt = Pe = 15.0000 MW` (power balance closed at every node)

If any of these are off by more than parts-per-thousand, the IC solver has a bug or D_hp got reintroduced incorrectly.

In [ ]:
r_ss = simulate_multishaft(np.array([0.0, 30.0]), np.array([15.0, 15.0]),
                            params=p, sample_dt_s=1.0, dispatch_fn=dispatch_fn)
print(f'SS hold @ 15 MW:')
print(f'  df          = {(r_ss.freq_hz[-1]-60)*1000:+.4f} mHz')
print(f'  omega_pt    = {r_ss.omega_pt_pu[-1]:.6f}')
print(f'  omega_hp    = {r_ss.omega_hp_pu[-1]:.5f} pu = {r_ss.speed_hp_rpm[-1]:.0f} rpm NGG')
print(f'  Pm_hp/Pm_pt = {r_ss.Pm_hp_mw[-1]:.4f} / {r_ss.Pm_pt_mw[-1]:.4f} MW (Pe = {r_ss.Pe_mw[-1]:.4f})')

## 3. Step 15 → 18 MW — the headline Tier C contribution

### Cell 4 — explanation

**This is the diagnostic that motivates Tier C.** Apply a 3 MW load step at t=2 s and watch four signals across three model tiers:

1. **Mechanical power Pm vs time** — Tier A/B show a single `Pm` curve; Tier C shows two distinct curves: `Pm_hp` (governor's heat-rate-equivalent output, driving the HP rotor input) and `Pm_pt` (power actually delivered to the PT/gen, via gas-path coupling). At SS these are equal; during the transient `Pm_hp` rises first, then `Pm_pt` catches up after the HP rotor spools.
2. **Frequency vs time** — Tier C's nadir should be slightly *deeper* than Tier A/B (typically 50-100 mHz worse) because the HP-rotor spool delay slows Pmech delivery to the PT rotor.
3. **HP rotor speed (NGG) vs time** — visible only in Tier C. Starts at 7,901 rpm (15 MW operating speed), overshoots ~8,632 rpm, settles at 8,450 rpm (18 MW operating speed). Should reach 95 % of the change in ~1.3 s.
4. **Valve position vs time** — should be similar across all three tiers (governor sees the same Pe error), with Tier C valve slightly higher during the dip because the governor pushes harder against the slower mechanical response.

**What's being solved:** three independent ODE integrations (TGOV1+fixes / GGOV1 / multishaft) over the same 30-s window with a step in Pe at t=2 s. All three use the same dispatch function for fuel post-processing.

**What's NOT being solved:** voltage dynamics (V held at 1 pu in all three), torsional modes (covered in Tier E), or HP-rotor combustor lag effects (subsumed into `T_b` in the gas-path block).

In [ ]:
load_t = np.array([0.0, 2.0, 30.0])
load_mw = np.array([15.0, 18.0, 18.0])

p_A = TGOV1Params()
p_B = GGOV1Params.lm2500_overrides()
p_C = MultishaftParams()

rA = simulate_tgov1(load_t, load_mw, params=p_A, sample_dt_s=0.01, dispatch_fn=dispatch_fn)
rB = simulate_ggov1(load_t, load_mw, params=p_B, sample_dt_s=0.01, dispatch_fn=dispatch_fn)
rC = simulate_multishaft(load_t, load_mw, params=p_C, sample_dt_s=0.01, dispatch_fn=dispatch_fn)

fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
ax = axes[0]
ax.plot(rA.t_s, rA.Pm_mw, 'C2-', lw=1.2, label='Pm Tier A (TGOV1+fixes)')
ax.plot(rB.t_s, rB.Pm_mw, 'C0-', lw=1.2, label='Pm Tier B (GGOV1)')
ax.plot(rC.t_s, rC.Pm_pt_mw, 'C3-', lw=1.2, label='Pm_pt Tier C (PT rotor — what gen sees)')
ax.plot(rC.t_s, rC.Pm_hp_mw, 'C3--', lw=1.0, alpha=0.7, label='Pm_hp Tier C (HP rotor input)')
ax.axhline(15, color='k', ls=':', lw=0.7); ax.axhline(18, color='k', ls=':', lw=0.7)
ax.set_ylabel('Power [MW]'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Step 15 → 18 MW at t=2 s — Tier A vs B vs C')

ax = axes[1]
ax.plot(rA.t_s, rA.freq_hz, 'C2-', lw=1.2, label='Tier A')
ax.plot(rB.t_s, rB.freq_hz, 'C0-', lw=1.2, label='Tier B')
ax.plot(rC.t_s, rC.freq_hz, 'C3-', lw=1.2, label='Tier C')
ax.axhline(60, color='k', ls=':', lw=0.7)
ax.set_ylabel('Frequency [Hz]'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(rC.t_s, rC.speed_hp_rpm, 'C4-', lw=1.2, label='HP rotor (NGG) — Tier C')
ax.axhline(9500, color='r', ls='--', lw=0.7, alpha=0.5, label='NGG full = 9500 rpm')
ax.axhline(4900, color='b', ls='--', lw=0.7, alpha=0.5, label='NGG idle = 4900 rpm')
ax.set_ylabel('HP rotor [rpm]'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[3]
ax.plot(rA.t_s, rA.P_valve_pu, 'C2-', lw=1.2, label='Tier A valve')
ax.plot(rB.t_s, rB.valve_pu, 'C0-', lw=1.2, label='Tier B valve')
ax.plot(rC.t_s, rC.valve_pu, 'C3-', lw=1.2, label='Tier C valve')
ax.set_ylabel('Valve [pu]'); ax.set_xlabel('Time [s]'); ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

# HP rotor dynamics summary
print(f'HP rotor: idle SS @ 15 MW = {rC.speed_hp_rpm[0]:.0f} rpm')
print(f'          peak during step = {rC.speed_hp_rpm.max():.0f} rpm')
print(f'          SS @ 18 MW       = {rC.speed_hp_rpm[-1]:.0f} rpm')
target_hp = rC.speed_hp_rpm[0] + 0.95 * (rC.speed_hp_rpm[-1] - rC.speed_hp_rpm[0])
above = np.where(rC.speed_hp_rpm >= target_hp)[0]
if len(above):
    print(f'          95% of step reached in {rC.t_s[above[0]] - 2.0:.3f} s — invisible in Tier A/B')

## 4. Voltage-exponent sensitivity (ZIP load model)

### Cell 5 — explanation

The ZIP load model decomposes a real load into constant-power, constant-current, and constant-impedance components. We capture the impedance/voltage sensitivity with a single exponent `pv`:

$$P_e = P_{demand} \cdot \left(\frac{V}{V_{ref}}\right)^{p_v} \cdot (1 + \alpha(\omega - 1))$$

- `pv = 0` → constant-P (purest electronic load, e.g. server PSUs with PFC)
- `pv = 1` → constant-I (rare in isolation)
- `pv = 2` → constant-Z (resistive heaters, motor at fixed speed)

In the scipy Tier C path, `V_term` is held at 1.0 pu (we are not solving the AVR/electrical algebraic loop here). The only way `pv` affects the result is indirectly through the omega-coupling term `(1 + α(ω-1))`, since the V-ratio is exactly 1.

**What to expect:** the three curves (pv = 0, 1, 2) should be **nearly identical**. Any visible spread is just numerical noise from the chunked integrator. If you want to see real ZIP effects, switch to the ANDES path where V dynamics live.

*This cell is therefore a NULL test*: it confirms that the scipy-path Tier C reduces to Tier B behavior when V is held constant, regardless of pv. Useful as a regression safety net.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
for pv, color in [(0.0, 'C0'), (1.0, 'C1'), (2.0, 'C2')]:
    p_pv = MultishaftParams(pv_exponent=pv)
    r = simulate_multishaft(load_t, load_mw, params=p_pv, sample_dt_s=0.01, dispatch_fn=dispatch_fn)
    ax.plot(r.t_s, r.freq_hz, color=color, lw=1.2, label=f'pv = {pv}')
ax.axhline(60, color='k', ls=':', lw=0.7)
ax.set_xlabel('Time [s]'); ax.set_ylabel('Frequency [Hz]')
ax.set_title('Tier C frequency response to 15→18 MW step, varying voltage exponent (V held = 1)')
ax.legend(); ax.grid(alpha=0.3)
plt.show()
print('Note: with V held at 1.0, pv only affects Pe through ω-coupling. With real V dynamics,')
print('pv > 0 would also reduce Pe when V dips — see the ANDES path for V-Q dynamics.')

## 5. Load17 7.4-hour full-window replay — four-model comparison

### Cells 6 & 7 — explanation

This is the ultimate side-by-side: run the same 7.4-hr / 8,927-point data-center demand profile through **all four tier models** under their default parameters, then plot identical panels for direct comparison.

**What's being solved:**

1. **Tier 0** (`TGOV1Params` with every fix disabled) — reproduces the original notebook's inline ODE. Used as the baseline.
2. **Tier A** (`TGOV1Params()` defaults) — TGOV1 + 6 bug fixes (anti-windup, valve rate limit, load damping, VMAX rebase, fuel-from-valve, event-driven integration).
3. **Tier B** (`GGOV1Params.lm2500_overrides()`) — proper aero governor topology (PI + accel limiter + temp limiter via LVG), Tact tuned for Woodward MkVIe.
4. **Tier C** (`MultishaftParams()`) — Tier B's GGOV1 + two-mass mechanical shaft.

**Wall-clock expectations (approximate, with my laptop):**
- Tier 0: ~5-10 s (no chunking, RK45 over the full window)
- Tier A: ~30-60 s (chunked + 6 fixes)
- Tier B: ~80 s (12-state RK45 chunked)
- Tier C: ~90-100 s (14-state RK45 chunked)

Each sim returns per-second sampled outputs over 26,781 s.

**What to expect in the print summary (representative — exact numbers from the run below):**
- `|Δf|max` magnitudes ranked roughly: **Tier 0 (374) < Tier B (460) ≈ Tier C < Tier A (478) mHz**. Tier A is largest because the valve rate limit (A2) is the dominant fidelity correction and it has no PI integrator to compensate. Tier B/C use GGOV1's PI which catches up faster.
- Fuel totals within ~5 % of each other (Tier B/C ~24.9 t, Tier A/0 ~26.3 t). The 5 % gap comes from `Kturb=1.5, Wfnl=0.2` in GGOV1 vs the implicit `Kturb=1` in TGOV1.

**What the panels show:**
- Panel 1 (Power): demand vs each model's Pm. They should track demand closely; Tier C's `Pm_pt` is what the generator actually sees.
- Panel 2 (Frequency): the four traces should overlap almost everywhere but diverge during the biggest load ramps. Tier 0 will be the calmest, Tier A the most agitated.
- Panel 3 (HP rotor): visible only in Tier C. Should oscillate between ~8,100 and 8,700 rpm tracking the demand swing (between ~16 MW and 19 MW operating points).
- Panel 4 (Cumulative fuel): four near-parallel lines, slight separation.

In [ ]:
load17 = pd.read_csv(ROOT / 'data' / 'load17.csv', sep=r'\s+', header=None, names=['time_s','demand_raw'])
load17['demand_mw'] = load17['demand_raw'] * 10.0
t_arr = load17['time_s'].values
d_arr = load17['demand_mw'].values

params_t0 = TGOV1Params(use_anti_windup=False, use_valve_rate_limit=False,
                        alpha_load_damping=0.0, fuel_from_valve=False, chunked=False,
                        vmax_pu=1.1, vmin_pu=0.0, T_comb_s=0.01)

import time as _t
_t0 = _t.time(); res_t0 = simulate_tgov1(t_arr, d_arr, params=params_t0, sample_dt_s=1.0, dispatch_fn=dispatch_fn); print(f'Tier 0: {_t.time()-_t0:.1f}s')
_t0 = _t.time(); res_tA = simulate_tgov1(t_arr, d_arr, params=TGOV1Params(), sample_dt_s=1.0, dispatch_fn=dispatch_fn); print(f'Tier A: {_t.time()-_t0:.1f}s')
_t0 = _t.time(); res_tB = simulate_ggov1(t_arr, d_arr, params=GGOV1Params.lm2500_overrides(), sample_dt_s=1.0, dispatch_fn=dispatch_fn); print(f'Tier B: {_t.time()-_t0:.1f}s')
_t0 = _t.time(); res_tC = simulate_multishaft(t_arr, d_arr, params=MultishaftParams(), sample_dt_s=1.0, dispatch_fn=dispatch_fn); print(f'Tier C: {_t.time()-_t0:.1f}s')

for label, r, get_freq, get_fuel in [
    ('Tier 0', res_t0, lambda r: r.freq_hz, lambda r: r.cum_fuel_kg),
    ('Tier A', res_tA, lambda r: r.freq_hz, lambda r: r.cum_fuel_kg),
    ('Tier B', res_tB, lambda r: r.freq_hz, lambda r: r.cum_fuel_kg),
    ('Tier C', res_tC, lambda r: r.freq_hz, lambda r: r.cum_fuel_kg),
]:
    fz = get_freq(r); fl = get_fuel(r)
    dfmax = max(abs(fz.max()-60), abs(60-fz.min())) * 1000
    print(f'{label}: |Δf|max = {dfmax:6.1f} mHz, fuel = {fl[-1]/1000:.3f} t')

In [ ]:
t_hr = res_tC.t_s / 3600
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

ax = axes[0]
ax.plot(t_hr, res_tC.Pe_demand_mw, 'C3-', lw=0.4, alpha=0.7, label='Demand')
ax.plot(t_hr, res_t0.Pm_mw, 'k:', lw=0.5, label='Tier-0')
ax.plot(t_hr, res_tA.Pm_mw, 'C2-', lw=0.5, label='Tier-A')
ax.plot(t_hr, res_tB.Pm_mw, 'C0-', lw=0.5, label='Tier-B')
ax.plot(t_hr, res_tC.Pm_pt_mw, 'C4-', lw=0.5, label='Tier-C (Pm_pt)')
ax.set_ylabel('Power [MW]'); ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)
ax.set_title('Load17 7.4-hr — Four-model comparison')

ax = axes[1]
ax.plot(t_hr, res_t0.freq_hz, 'k:', lw=0.4, label='Tier-0')
ax.plot(t_hr, res_tA.freq_hz, 'C2-', lw=0.4, label='Tier-A')
ax.plot(t_hr, res_tB.freq_hz, 'C0-', lw=0.4, label='Tier-B')
ax.plot(t_hr, res_tC.freq_hz, 'C4-', lw=0.4, label='Tier-C')
ax.axhline(60, color='gray', ls=':', lw=1)
ax.set_ylabel('Frequency [Hz]'); ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(t_hr, res_tC.speed_hp_rpm, 'C5-', lw=0.4, label='HP rotor (Tier-C only)')
ax.axhline(9500, color='r', ls=':', lw=0.5, alpha=0.5)
ax.axhline(4900, color='b', ls=':', lw=0.5, alpha=0.5)
ax.set_ylabel('HP rotor [rpm]'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[3]
ax.plot(t_hr, res_t0.cum_fuel_kg/1000, 'k:', lw=1.0, label='Tier-0')
ax.plot(t_hr, res_tA.cum_fuel_kg/1000, 'C2-', lw=1.0, label='Tier-A')
ax.plot(t_hr, res_tB.cum_fuel_kg/1000, 'C0-', lw=1.0, label='Tier-B')
ax.plot(t_hr, res_tC.cum_fuel_kg/1000, 'C4-', lw=1.0, label='Tier-C')
ax.set_ylabel('Cumulative fuel [t]'); ax.set_xlabel('Time [hours]'); ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Summary

Captured in `docs/tier_plan.md` Tier C section.

**Key novelty:** HP-rotor speed transient is now resolved as a separate dynamic. The ~0.3-1 s HP-spool time constant is invisible in Tier A/B (which lump it into the single rotor) but matters for:
- Interpreting fuel-step response correctly (Pm_hp leads Pm_pt by the spool delay)
- Surge-margin analysis (HP rotor overshoot during fast load steps can stress the compressor)
- Setting acceleration-controller gains (Ka, aset in GGOV1) against realistic dynamics

**What Tier C does NOT do:**
- Voltage / reactive-power dynamics (V held at 1.0 pu)
- Field-flux / E' dynamics (no GENROU)
- AVR / exciter dynamics (perfect AVR assumed)
- Torsional shaft modes (covered in Tier E)

For full V-Q + AVR dynamics with proper algebraic-loop solving, use the ANDES path in `gas_plant_andes/` (upgrade to GENROU + EXST1 + GGOV1 still pending — currently uses GENCLS + TGOV1).